In [1]:
import Dynasplit as ds
import MDAnalysis as mda
import numpy as np

/opt/anaconda3/envs/Fresh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load an example Universe

In [2]:
u = mda.Universe("./test_traj/290k_end.data", "./test_traj/sampled_100_frames.dcd")

/opt/anaconda3/envs/Fresh/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


## Create an instance of the Dynnasplit class with the universe

In [3]:
splitter = ds.Dynasplit(u)
fast = ds.Dynasplit(u)

In [4]:
splitter.calc_com(slower=True, atom_types="type 1 2")
fast.calc_com()

729 molecules selected with 12 atoms each.


Centering molecules: 100it [00:03, 27.07it/s]


729 molecules selected with 12 atoms each.


Centering molecules: 100it [00:00, 1234.90it/s]


## The main feature of the class is decompose which produces an independent rotational and translational trajectory.

It can be called with .decompose(), if the indices and masses information is stored in the universe then no parameters are required, but specific indices and masses can be input. Currently only trajectories of the same molecules are supported.

In [5]:
splitter.decompose()

729 molecules selected with 12 atoms each.


Centering molecules: 100it [00:00, 1064.08it/s]


Alternatively indices and masses can be specified as numpy arrays, both indices and masses should be shape (n_molecules, n_atoms_per_molecule). 

In [6]:
indices = np.arange(0, 8748, 1).reshape(-1, 12)
indices.shape

(729, 12)

In [7]:
indices = np.arange(0, 8748, 1).reshape(-1, 12)
masses = np.tile(np.array([12, 12, 12, 12, 12, 12, 1, 1, 1, 1, 1, 1]), (729, 1))
splitter.decompose(indices=indices, masses=masses)

729 molecules selected with 12 atoms each.


Centering molecules: 100it [00:00, 1064.10it/s]


Alternatively MDAnalaysis can be used in which case only the atom types need to specified, in which case bond information is required

In [8]:
splitter.decompose(slower=True, atom_types="type 1 2 3 4 5 6 7 8 9 10 11 12")

729 molecules selected with 12 atoms each.


Centering molecules: 100it [00:03, 25.29it/s]


The trajectories can then be accessed via .rot_traj & .trans_traj

In [9]:
splitter.rot_traj[0]

array([[-2.97902536, 17.90842819, -9.07791805],
       [-3.99814177, 18.46211624, -9.85601425],
       [-5.06566095, 19.20675278, -9.29435253],
       ...,
       [28.31654358, -2.41946459, 11.11115837],
       [26.25740051, -3.36248922, 11.51376343],
       [25.98564148, -5.3635335 , 13.09955215]], shape=(8748, 3))

In [10]:
splitter.trans_traj[0]

array([[-2.97902536, 17.90842819, -9.07791805],
       [-3.99814177, 18.46211624, -9.85601425],
       [-5.06566095, 19.20675278, -9.29435253],
       ...,
       [28.31654358, -2.41946459, 11.11115837],
       [26.25740051, -3.36248922, 11.51376343],
       [25.98564148, -5.3635335 , 13.09955215]], shape=(8748, 3))

The trajectories can also written to DCD files with .write